# Real Historical Crash-Period Stress Test (2008 GFC / 2020 COVID / 2022 Bear Market)

Epic 13: answers `README.md`'s Known Gap "Has it been validated against real 2008/2020/2022
history?" with actual historical data instead of only synthetic crash-shaped test fixtures.

**Scope boundary, read this first**: several of the currently-configured tickers didn't exist
during these crashes (e.g. `ARM` IPO'd 2023, `PLTR` 2020, `ASTS`/`CPA`/`SWMR` are all recent),
so this can't literally backtest `portfolio1`/`portfolio2`/`portfolio3` exactly as configured
through 2008. Instead it uses a **long-history, liquid ETF proxy universe** (17 tickers, all
confirmed to have real price data back to 2005, see `crash_test_daily_prices.pkl` built by the
companion fetch step) with each of this project's two REAL risk regimes (`portfolio1`'s monthly
config, `portfolio2`'s weekly config, both loaded straight from `config.yaml` via
`daily_runner.load_config()`, not hand-typed) to answer a narrower, honest question: **does this
project's risk-control machinery (regime filter, volatility targeting, stop-loss, circuit
breaker) actually reduce drawdown and shorten recovery time under real historical stress**,
compared to the same momentum signal with every defensive mechanism turned off. This validates
the risk-control *mechanism*, not a claim about what the exact current portfolios would have
returned in 2008.

In [ ]:
# Package is pip-installed editable, no sys.path hacking needed
import os
import re
import tempfile

import numpy as np
import pandas as pd

from momentum_trading.daily_runner import load_config
from momentum_trading.backtest.momentum_backtest import run_custom_backtest, BacktestConfig
from momentum_trading.core.strategy_signals import generate_strategy_monthly_picks
from momentum_trading.core.functions_quant_extensions import compute_drawdown_episodes

## 1. Load the two real risk regimes from `config.yaml`

In [ ]:
# Confirmed via a direct yfinance check (Story 13.1) that all 17 have real data back to 2005.
# HYG (inception 2007) deliberately excluded, LQD already covers credit exposure.
PROXY_TICKERS = [
    "SPY", "QQQ", "DIA", "XLK", "XLF", "XLE", "XLI", "XLP", "XLU", "XLV", "XLY",
    "GLD", "TLT", "IEF", "SHY", "LQD", "IWM",
]

config = load_config()
portfolio1_cfg = config["portfolios_resolved"]["portfolio1"]["cfg"]  # monthly regime
portfolio2_cfg = config["portfolios_resolved"]["portfolio2"]["cfg"]  # weekly regime

# The real config.yaml ships the circuit breaker OFF by default (max_portfolio_drawdown_pct:
# 0.0, an explicit "not yet turned on" choice, confirmed by reading config.yaml directly), so
# "as configured today" would never exercise it at all. To actually test the circuit breaker
# MECHANISM, the "full" variant below uses a representative 0.20 (-20% halts new entries),
# matching the illustrative value config.yaml's own comments use, not a claim about what's live
# in production right now.
CIRCUIT_BREAKER_PCT = 0.20

# Epic 14 (Daniel & Moskowitz 2016 "Momentum Crashes"): a THIRD variant, "full" plus an EXTRA
# multiplicative derate on top of regime_scalar * vol_scalar, only when the benchmark is BOTH in
# a sustained decline over momentum_crash_lookback_days AND currently volatile
# (regime_vol_threshold's high_vol, reused not duplicated). Requires regime_vol_threshold to be
# set too; real config.yaml ships that null (disabled) by default, so this uses a representative
# 0.25 (25% annualized realized vol, a common "elevated vol" threshold), same "illustrative, not
# a claim about production" caveat as CIRCUIT_BREAKER_PCT above.
REGIME_VOL_THRESHOLD = 0.25
MOMENTUM_CRASH_LOOKBACK_DAYS = 504  # ~24 months, Daniel & Moskowitz's own empirical choice
MOMENTUM_CRASH_DERATE = 0.5


def _regime_kwargs(cfg: BacktestConfig, overrides: dict | None = None) -> dict:
    """Every real BacktestConfig field from a loaded portfolio's cfg, as a plain dict suitable
    for run_custom_backtest(**kwargs), the same reuse pattern Epic 12 established."""
    kwargs = dict(cfg.__dict__)
    if overrides:
        kwargs.update(overrides)
    return kwargs


# baseline = naive momentum, every defensive mechanism neutralized (a fixed stop can't be fully
# disabled portfolio-wide in the backtest engine, only pushed out of range via stop_loss_pct).
_BASELINE_OVERRIDES = {
    "use_regime_filter": False,
    "target_portfolio_vol": 10.0, "min_gross_exposure": 1.0, "max_gross_exposure": 1.0,
    "stop_loss_pct": 0.95,
    "use_trailing_stop": False,
    "max_portfolio_drawdown_pct": 0.0,
    "use_correlation_spike_regime": False,
    "use_correlation_penalty": False,
}

_MOMENTUM_CRASH_OVERRIDES = {
    "max_portfolio_drawdown_pct": CIRCUIT_BREAKER_PCT,
    "regime_vol_threshold": REGIME_VOL_THRESHOLD,
    "momentum_crash_lookback_days": MOMENTUM_CRASH_LOOKBACK_DAYS,
    "momentum_crash_derate": MOMENTUM_CRASH_DERATE,
}

regimes = {
    "monthly (portfolio1-style)": {
        "full": _regime_kwargs(portfolio1_cfg, {"max_portfolio_drawdown_pct": CIRCUIT_BREAKER_PCT}),
        "baseline (no risk mgmt)": _regime_kwargs(portfolio1_cfg, _BASELINE_OVERRIDES),
        "full + momentum_crash_protection": _regime_kwargs(portfolio1_cfg, _MOMENTUM_CRASH_OVERRIDES),
    },
    "weekly (portfolio2-style)": {
        "full": _regime_kwargs(portfolio2_cfg, {"max_portfolio_drawdown_pct": CIRCUIT_BREAKER_PCT}),
        "baseline (no risk mgmt)": _regime_kwargs(portfolio2_cfg, _BASELINE_OVERRIDES),
        "full + momentum_crash_protection": _regime_kwargs(portfolio2_cfg, _MOMENTUM_CRASH_OVERRIDES),
    },
}

for regime_name, variants in regimes.items():
    for variant_name, kwargs in variants.items():
        print(f"{regime_name} / {variant_name}: holding_period={kwargs['holding_period']} "
              f"lookback_period={kwargs['lookback_period']} top_n={kwargs['top_n']} "
              f"use_regime_filter={kwargs['use_regime_filter']} "
              f"stop_loss_pct={kwargs['stop_loss_pct']} "
              f"max_portfolio_drawdown_pct={kwargs['max_portfolio_drawdown_pct']} "
              f"momentum_crash_lookback_days={kwargs.get('momentum_crash_lookback_days')}")

## 2. Load the cached proxy-universe price history and define the three real crash windows

In [ ]:
# Built by the companion fetch step (Story 13.1), relative to this notebook's own directory.
daily_prices = pd.read_pickle("crash_test_daily_prices.pkl")
print(daily_prices.shape, daily_prices.index.min(), "->", daily_prices.index.max())

# Windows chosen off real, well-documented crash dates. Pre-window history needed for lookback
# warm-up (monthly regime ~12mo, weekly ~1mo) is already covered by the 2005-01-01 fetch start.
CRASH_WINDOWS = {
    "2008 GFC":   ("2007-10-01", "2009-06-30"),  # Oct 2007 peak -> partial 2009 recovery
    "2020 COVID": ("2020-01-01", "2020-12-31"),  # sharp crash + V-shaped recovery
    "2022 Bear":  ("2022-01-01", "2022-12-31"),  # slower, rate-driven decline
}

## 3. Run all 12 backtests (2 regimes x 2 variants x 3 crash periods)

The momentum signal (`generate_strategy_monthly_picks()`) depends only on `strategy_type`/
`lookback_period`/`top_n`/liquidity filters, none of which differ between the baseline and full
variant of a given regime, so picks are computed ONCE per regime and reused for both variants,
the same shared-signal principle Epic 12 established for live/backtest parity.

In [ ]:
log_dir = tempfile.mkdtemp(prefix="crash_test_logs_")
results = []

for regime_name, variants in regimes.items():
    real_cfg = portfolio1_cfg if "monthly" in regime_name else portfolio2_cfg
    monthly_picks = generate_strategy_monthly_picks(
        daily_prices, PROXY_TICKERS, real_cfg, real_cfg.lookback_period, real_cfg.top_n,
    )

    for period_name, (start, end) in CRASH_WINDOWS.items():
        picks_window = monthly_picks[(monthly_picks.index >= start) & (monthly_picks.index <= end)]
        # Bounded only at the END, not the start: a real, confirmed methodology bug found while
        # validating momentum_crash_lookback_days (Epic 14). Truncating the START discarded the
        # 2005+ history a 504-trading-day (~24mo) lookback needs, silently making the condition
        # unable to ever fire, identical in nature to the compute_required_lookback_days() gap
        # fixed in execution/live_signal.py for the same reason. The simulation itself still only
        # runs within [start, end] (governed by picks_window's own date range via
        # run_risk_managed_backtest()'s own sim_start_date logic), this only widens what's
        # available for regime/vol/momentum-crash lookback computation.
        prices_for_backtest = daily_prices[daily_prices.index <= end]
        prices_window = daily_prices[(daily_prices.index >= start) & (daily_prices.index <= end)]
        spy_bh_return = prices_window["SPY"].iloc[-1] / prices_window["SPY"].iloc[0] - 1.0

        for variant_name, kwargs in variants.items():
            safe_name = re.sub(r"[^a-zA-Z0-9]+", "_", f"{regime_name}_{variant_name}_{period_name}")
            run_kwargs = dict(kwargs)
            run_kwargs["log_file_path"] = os.path.join(log_dir, f"{safe_name}.txt")

            backtest_df = run_custom_backtest(picks_window, prices_for_backtest, **run_kwargs)

            row = {
                "regime": regime_name, "variant": variant_name, "period": period_name,
                "spy_buy_hold_return": spy_bh_return,
            }

            if backtest_df.empty:
                row.update(total_return=np.nan, max_drawdown=np.nan, worst_dd_recovery_days=np.nan,
                           cb_trips=0, sharpe=np.nan, sortino=np.nan, calmar=np.nan)
                results.append(row)
                continue

            tearsheet = backtest_df.attrs.get("tearsheet", {})
            cum = backtest_df["Portfolio Cumulative Return"]
            episodes = compute_drawdown_episodes(cum)
            worst_recovery = episodes.loc[episodes["trough_pct"].idxmin(), "recovery_days"] if not episodes.empty else 0.0

            with open(run_kwargs["log_file_path"]) as f:
                cb_trips = f.read().count("CIRCUIT BREAKER TRIPPED")

            row.update(
                total_return=cum.iloc[-1] - 1.0,
                max_drawdown=tearsheet.get("MaxDrawdown", np.nan),
                worst_dd_recovery_days=worst_recovery,
                cb_trips=cb_trips,
                sharpe=tearsheet.get("Sharpe", np.nan),
                sortino=tearsheet.get("Sortino", np.nan),
                calmar=tearsheet.get("Calmar", np.nan),
            )
            results.append(row)

results_df = pd.DataFrame(results)
results_df

## 4. Results: baseline (no risk management) vs. full (as-configured), per crash period

In [ ]:
cols = ["total_return", "max_drawdown", "worst_dd_recovery_days", "cb_trips",
        "sharpe", "sortino", "calmar", "spy_buy_hold_return"]
for period_name in CRASH_WINDOWS:
    print(f"\n=== {period_name} ===")
    sub = results_df[results_df["period"] == period_name].set_index(["regime", "variant"])
    display(sub[cols])